# Rules-based backtest

This notebook runs only the transparent configurable-frequency momentum strategy. The optional machine-learning comparison is in `Machine Learning Trade Model/notebooks/ml_trade_model.ipynb`.

In [ ]:
import sys
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next((path for path in candidates if (path / 'src' / 'trading').is_dir()), None)
if project_root is None:
    raise RuntimeError('Could not locate the project root containing src/trading')
sys.path.insert(0, str(project_root / 'src'))
from trading.config import load_config
from trading.data import load_prices
from trading.risk import portfolio_risk_diagnostics, risk_diagnostics
from trading.strategy import weekly_momentum_weights
from trading.backtest import format_performance_report, run_backtest
from trading.visualization import plot_diagnostics, plot_monthly_cash_flow, plot_periodic_returns
config = load_config(project_root / 'config.yaml')
prices = load_prices(project_root / 'data' / 'processed' / 'prices.parquet')
plot_start = None  # Set to '2020-01-01' to inspect a specific range.
plot_end = None    # Set to '2020-12-31'; leave both as None for full history.
risk_profile = 'balanced'  # Also try 'conservative' or 'aggressive'.
profile = config.values.get('risk_profiles', {}).get(risk_profile, {})
excluded_symbols = [symbol for asset_class in ('forex', 'bonds', 'commodities') for symbol in config.values.get('asset_classes', {}).get(asset_class, [])]
benchmark_fallback = True  # Hold SPY when no eligible momentum candidates exist.
print(f'Loaded {prices.shape[1]:,} symbols from {prices.index.min().date()} to {prices.index.max().date()}')
print(f"Frequency: {config['frequency']} | Risk profile: {risk_profile}")


In [ ]:
risk = risk_diagnostics(prices, config['benchmark'], config['forecast_horizon_days'], config['volatility_days'])
weights = weekly_momentum_weights(
    prices, config['benchmark'], profile.get('top_n', config['top_n']), config['lookback_days'], profile.get('skip_days', config['skip_days']),
    config['volatility_days'], profile.get('max_position_weight', config['max_position_weight']), profile.get('max_equity_exposure', config['max_equity_exposure']),
    config['defensive_exposure'], profile.get('stop_loss_pct', config['stop_loss_pct']), config['drawdown_start'],
    config['drawdown_full_defense'], volatility_target=profile.get('volatility_target', config['volatility_target']), excluded_symbols=excluded_symbols, benchmark_fallback=benchmark_fallback,
    rebalance_frequency=config['frequency']
)
ledger, holdings, trades = run_backtest(
    prices, weights, config['initial_capital'], config['commission_bps'],
    config['half_spread_bps'], config['slippage_bps'], config['market_impact_bps'],
    execution_lag_days=config['signal_execution_lag_days']
)
portfolio_risk = portfolio_risk_diagnostics(holdings, prices, config['benchmark'])
print('RULES-BASED MODEL')
print(format_performance_report(ledger, config['initial_capital']))
print(f'Trades recorded: {len(trades):,}')


In [ ]:
plot_diagnostics(ledger, trades, portfolio_risk, prices[config['benchmark']])


In [ ]:
plot_diagnostics(ledger, trades, portfolio_risk, prices[config['benchmark']], start=plot_start, end=plot_end)


In [ ]:
plot_periodic_returns(ledger, period='Y')
plot_periodic_returns(ledger, period='M')
plot_periodic_returns(ledger, period='5Y')
plot_monthly_cash_flow(trades, year=2020)


In [ ]:
trades.style.format({
    'weight_before': '{:.3%}', 'weight_after': '{:.3%}', 'weight_change': '{:.3%}',
    'execution_price': '${:,.3f}', 'notional': '${:,.3f}', 'estimated_cost': '${:,.3f}',
    'realized_profit_loss': '${:,.3f}', 'net_profit_loss': '${:,.3f}',
    'profit_loss_pct': '{:.3%}', 'cumulative_net_profit_loss': '${:,.3f}',
    'cash_flow': '${:,.3f}', 'cumulative_cash_flow': '${:,.3f}',
})
